In [ ]:
# ruff: noqa: T201, T203, ERA001, E402
from __future__ import annotations

import json
import logging
import os
from itertools import product
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

while Path.cwd().name in ["notebooks", "inference"]:
    os.chdir("..")

import src.prompts.system as system_prompts
from config.config import DATA_DIR
from config.runs_vars import SUBSET_TO_DATASET_MAP
from src.evaluate import (
    get_predictions_with_gt,
    plot_usage_histograms,
    save_analysis_results,
    store_run_metrics_df,
)
from src.formatting import (
    format_predictions_run_path,
    format_storing_pathes_from_run_path,
)
from src.LLM_servers.gemini import GeminiApiServer
from src.LLM_servers.open_router import OpenRouterServer
from src.LLM_servers.openai import OpenAIServer
from src.processing import parallel_samples_process, try_load_partial_results, try_load_precomputed_oracle_pairs
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
    prompt_uri,
)
from src.utils import expand_setups, save_run_results, save_token_usage_stats

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

### Run experiments


In [ ]:
MAX_WORKERS = 1500
MODELS = [
    # "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-1.5-flash",
    # "google|gemma-3-12b-it",
    # "gemini-2.5-pro"
    # "gemini-2.5-flash-lite",
    # "gpt-4.1-nano"
]  # "gpt-4o-mini", "gemini-2.5-flash-preview-04-17", "gemini-2.0-flash-lite", "gemini-1.5-flash", "gemini-2.0-flash"

DATASETS_MAP = {
    # "bioml-2024": ["omim-ordo", "snomed-ncit.pharm", "ncit-doid", "snomed-fma.body"][:0],  # "snomed-ncit.neoplas"],  # "snomed-fma.body"]],
    # "largebio": ["fma-nci", "fma-snomed"][1:],  # "snomed-nci"],  # , ],
    "anatomy": ["human-mouse"],
}
SUFFIX = "_reduced"
# from src.constants import BinaryOutputFormat, BinaryOutputFormatWithReasoning

# EXP_NAMES_MAP = {"absent": {}, "v1": few_shots}
# EXP_NAMES_MAP = {
#     "no-reasoning": BinaryOutputFormat,
#     "with-reasoning": BinaryOutputFormatWithReasoning,
# }
EXPERIMENT_TYPE = "n_few_shots"  # "few-shot", "reasoning"
EXP_NAMES_MAP = {
    f"v{version}/{n}": n
    for n in (0, 1, 2, 3, 5, 8, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)[::-1]
    for version in range(5, 10)
}

onto_src, onto_tgt = None, None

PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        prompt_direct_entity,
        prompt_direct_entity_ontological,
        prompt_direct_entity_with_synonyms,
        prompt_sequential_hierarchy,
        prompt_sequential_hierarchy_ontological,
        prompt_sequential_hierarchy_with_synonyms,
        prompt_uri,
    ]
}

SETUPS = []
for DATASET, SUBSETS in DATASETS_MAP.items():
    for SUBSET, EXP_NAME, MODEL in product(SUBSETS, EXP_NAMES_MAP.keys(), MODELS):
        setup = {
            "subset": SUBSET,
            "model": MODEL,
            "experiment_name": EXP_NAME,
            "max_workers": MAX_WORKERS,
            "suffix": SUFFIX,
            "prompt_functions": list(PROMPT_FUNCTIONS_MAP.keys()),
            "experiment_type": EXPERIMENT_TYPE,
        }
        SETUPS.append(setup)


In [ ]:
BASE_SETUP = {
    "subset": "human-mouse",
    "model": "gemini-2.5-pro",
    # "model": "mistralai|ministral-3b-2512",
    # "model": "mistralai/mistral-small-3.2-24b-instruct",
    # "model":"qwen|qwen3-14b", # issue with response format
    # "model": "meta-llama|llama-4-scout",
    # "experiment_name": "v0/0",
    "max_workers": 1000,
    "suffix": "_reduced",
    # "experiment_type": "n_few_shots",
    "experiment_type": "open-weight-models",
    # "experiment_type": "thinking",
    # "experiment_type": "benchmark-usage",
    # "experiment_type": "leack_check",
    "prompt_functions": ["direct_entity", "direct_entity_with_synonyms"],
    "sleep_time": 0.0,
    "model_params": {},
}


SETUPS = [
    *[
        {
            "subset": [
                "human-mouse",
                "ncit-doid",
                "omim-ordo",
                "snomed-fma.body",
                "snomed-ncit.neoplas",
                "snomed-ncit.pharm",
                "fma-nci",
                "fma-snomed",
                "snomed-nci",
            ],
            "model": model,
            "experiment_name": model,
            # "experiment_type": "thinking",
            "experiment_type": "open-weight-models",
            # "sleep_time": 20.0,
            # "experiment_name": [f"thinking_budget={x}" for x in [512]],
            "prompt_functions": ["direct_entity", "direct_entity_with_synonyms"],
            "model_params": {"custom_parsing": True, "use_openrouter": True},
        }
        for model in [
            # "minimax|minimax-m2.5",
            # "deepseek|deepseek-v3.2",
            # "z-ai|glm-5",
            # "openai|gpt-oss-20b",
            # "qwen|qwen3-32b",
            # "amazon|nova-micro-v1",
            # "meta-llama|llama-3.3-70b-instruct",
            # "bytedance-seed|seed-2.0-mini",
            # "qwen|qwen3.5-35b-a3b",
            # "qwen|qwen3.5-122b-a10b",
            "qwen|qwen3.5-flash-02-23",
            # "google|gemini-3.1-flash-lite-preview",
            # "anthropic|claude-sonnet-4.6",
            # "anthropic|claude-opus-4.6",
            # "anthropic|claude-haiku-4.5",
            # "openai|gpt-5.4",
            # "qwen|qwen3-235b-a22b-2507",
            # "qwen|qwen3-next-80b-a3b-instruct",  # not stable response format
            # "openai|gpt-oss-120b",  # no response format
            # "google|gemini-2.5-pro",
            # "gemini-2.5-flash",
            # "gemini-2.5-flash-lite",
            # "x-ai|grok-4.1-fast",
            # "deepseek|deepseek-chat-v3-0324",
            # "mistralai|mistral-nemo",
            # "meta-llama|llama-4-maverick",
            # "mistralai|ministral-3b-2512",
            # "mistralai|ministral-14b-2512",
            # "mistralai|mistral-large-2512",
            # "meta-llama|llama-4-scout",
            # "google|gemma-3-12b-it",
        ]
    ],
    # {
    #     "model": [
    #         # "gemini-2.5-flash",
    #         # "gemini-2.5-flash-lite",
    #         # "gemini-2.0-flash",
    #         # "gemini-2.0-flash-lite",
    #         # "openai|gpt-4o-mini",
    #         "mistralai|mistral-small-3.2-24b-instruct",
    #         "meta-llama|llama-3.1-70b-instruct",
    #         # "qwen|qwen3-vl-8b-instruct",
    #     ],
    #     "experiment_name": ["0"],
    #     # "model_params": {"custom_parsing": True, "use_openrouter": True},
    #     # "prompt_functions": [
    #     #     "direct_entity",
    #     #     "direct_entity_ontological",
    #     #     "direct_entity_with_synonyms",
    #     #     "sequential_hierarchy",
    #     #     "sequential_hierarchy_ontological",
    #     #     "sequential_hierarchy_with_synonyms",
    #     # ],
    # }
    # *[
    #     {
    #         "subset": ["human-mouse", "snomed-ncit.pharm", "fma-nci"][:],
    #         "model": model,
    #         "experiment_name": model,
    #         "model_params": {"custom_parsing": False, "use_openrouter": True},
    #     }
    #     for model in [
    #         # "qwen|qwen3-235b-a22b-2507",
    #         "qwen|qwen3-next-80b-a3b-instruct", # not stable response format
    #         "openai|gpt-oss-120b", # no respnse format
    #         "x-ai|grok-4.1-fast",
    #         # "deepseek|deepseek-chat-v3-0324",
    #         "mistralai|mistral-nemo",
    #         # "meta-llama|llama-4-maverick",
    #         "mistralai|ministral-3b-2512",
    #         "mistralai|ministral-14b-2512",
    #         "mistralai|mistral-large-2512",
    #         "meta-llama|llama-4-scout",
    #         # "google|gemma-3-12b-it",
    #     ]
    # ],
    # {
    #     "model": "gemini-3-pro-preview",
    #     "max_workers": 100,
    #     "subset": ["human-mouse"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(1) for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41)[:3]],
    #     "prompt_functions": ["direct_entity_with_synonyms"],
    #     "model_params": {"use_openrouter": True},
    # },
    # {
    #     "subset": ["human-mouse", "snomed-ncit.pharm"],
    #     "model": "gemini-2.5-flash-lite",
    #     "max_workers": 350,
    #     "sleep_time": 60.0,
    #     "experiment_name": [f"thinking_budget={x}" for x in [0, 512, 1024]],
    #     "prompt_functions": ["direct_entity", "direct_entity_with_synonyms"],
    #     "experiment_type": "thinking",
    # },
    # {
    #     "subset": ["human-mouse", "snomed-ncit.pharm"],
    #     "model": "openai|gpt-5-nano",
    #     # "max_workers": 350,
    #     # "sleep_time": 60.0,
    #     "experiment_name": [f"reasoning_effort={x}" for x in ["minimal", "low", "medium", "high"]],
    #     "prompt_functions": ["direct_entity", "direct_entity_with_synonyms"],
    #     "experiment_type": "thinking",
    # },
    # {
    # "model":"gemini-2.5-pro",
    # "max_workers": 100,
    # "subset": ["human-mouse", "snomed-ncit.pharm"],
    # "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41)],
    # "prompt_functions": ["direct_entity_with_synonyms"],
    # },
    # {
    #     "model": "gemini-2.5-pro",
    #     "max_workers": 5,
    #     "subset": ["human-mouse", "snomed-ncit.pharm"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (26, 33, 41)],
    #     "prompt_functions": ["direct_entity_with_synonyms"],
    #     "sleep_time": 20,
    # },
    #     {
    #     "model": "gemini-2.5-pro",
    #     "max_workers": 4,
    #     "subset": ["human-mouse", "snomed-ncit.pharm"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (15, 20)],
    #     "prompt_functions": ["direct_entity_with_synonyms"],
    #     "sleep_time": 15,
    # },
    # {
    #     "model": "gemini-2.5-pro",
    #     "max_workers": 2,
    #     "subset": ["human-mouse", "snomed-ncit.pharm"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (3, 5, 8, 11)],
    #     "prompt_functions": ["direct_entity_with_synonyms"],
    #     "sleep_time": 8,
    # },
    # {
    #     "model": "gemini-2.5-pro",
    #     "max_workers": 1,
    #     "subset": ["human-mouse", "snomed-ncit.pharm"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (0, 1, 2)],
    #     "prompt_functions": ["direct_entity_with_synonyms"],
    #     "sleep_time": 4,
    # },
    # {
    # "subset": ["human-mouse", "snomed-ncit.pharm"],
    # "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41)],
    # "prompt_functions": ["direct_entity_with_synonyms"],
    # }
    # {
    # "subset": ["human-mouse"],
    # "model": ["gemini-2.5-flash-lite"],
    # "experiment_name": [f"v{v}/{n}" for v in range(5) for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41)[::-1]],
    # },
    # {
    # "subset": ["snomed-ncit.pharm", "fma-nci"],
    # "model": ["gemini-2.5-flash-lite"],
    # "experiment_name": [f"v{v}/{n}" for v in range(5) for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41)[::-1]],
    # },
    # {
    #     "subset": ["human-mouse"],
    #     "model": ["gemini-2.5-flash"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (0, 1, 2, 3, 5)],
    #     "max_workers": [140],
    # },
    # {
    #     "subset": ["human-mouse"],
    #     "model": ["gemini-2.5-flash"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (8, 11, 15)],
    # },
    # {
    #     "subset": ["snomed-ncit.pharm", "fma-nci"],
    #     "model": ["gemini-2.5-flash"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (0, 1, 2, 3)],
    #     "max_workers": [200],
    # },
    # {
    #     "subset": ["snomed-ncit.pharm", "fma-nci"],
    #     "model": ["gemini-2.5-flash"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (5, 8, 11, 15)],
    #     "max_workers": [444],
    # },
    # {
    #     "subset": ["snomed-ncit.pharm", "fma-nci"],
    #     "model": ["gemini-2.5-flash"],
    #     "experiment_name": [f"v{v}/{n}" for v in range(3) for n in (20, 26, 33, 41)],
    # },
]

SETUPS = expand_setups(SETUPS, BASE_SETUP, exclude_keys=["prompt_functions"])

len(SETUPS)

In [ ]:
for setup in SETUPS[:]:
    SUBSET = setup["subset"]
    DATASET = SUBSET_TO_DATASET_MAP[SUBSET]
    MODEL = setup["model"]
    EXP_NAME = setup["experiment_name"]
    MAX_WORKERS = setup["max_workers"]
    SUFFIX = setup["suffix"]
    EXPERIMENT_TYPE = setup["experiment_type"]

    setup["model_params"] = {
        **setup.get("model_params", {}),
        **({p.split("=")[0]: p.split("=")[1] for p in EXP_NAME.split("&")} if ("=" in EXP_NAME) else {}),
    }
    print(setup["model_params"])

    PROMPT_FUNCTIONS = {name: PROMPT_FUNCTIONS_MAP[name] for name in setup["prompt_functions"]}

    llm_oracle = (
        GeminiApiServer(**setup.get("model_params", {}))
        if MODEL.startswith("gemini") and not setup.get("model_params", {}).get("use_openrouter", False)
        else OpenRouterServer(
            max_tokens=1000, api_key=os.getenv("OPENROUTER_MODELS_BENCHMARKING_2"), **setup.get("model_params", {})
        )
    )

    llm_oracle.add_system_context(system_prompts.BASELINE_INITIALIZATION_MESSAGE)

    run_path = format_predictions_run_path(DATASET, SUBSET, MODEL, EXPERIMENT_TYPE, exp_spec=EXP_NAME)
    if EXPERIMENT_TYPE == "n_few_shots":
        version, n_few_shots = EXP_NAME.split("/")
        n_few_shots = int(n_few_shots)

        few_shot_samples_path = DATA_DIR / f"preparation/{DATASET}/{SUBSET}/{version}.json"
        few_shots = json.load(few_shot_samples_path.open())["OpenAI"]

    print(MODEL, SUBSET, EXP_NAME)

    for prompt_name, prompt_function in PROMPT_FUNCTIONS.items():
        if EXPERIMENT_TYPE == "n_few_shots":
            llm_oracle.clear_context()
            llm_oracle.add_system_context(system_prompts.BASELINE_INITIALIZATION_MESSAGE)

            for sample in few_shots[prompt_function.__name__][: 2 * n_few_shots]:
                llm_oracle.add_context(sample["input"], "user")
                llm_oracle.add_context(sample["output"], "assistant")

        prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
            run_path, SUBSET, MODEL, prompt_name, suffix=SUFFIX
        )
        oracle_candidate_pairs = try_load_precomputed_oracle_pairs(DATASET, SUBSET, prompt_function, suffix=SUFFIX)
        partial_results = try_load_partial_results(prediction_path)

        results, tokens_usage, confidences = parallel_samples_process(
            oracle_candidate_pairs,
            llm_oracle,
            onto_src=None,
            onto_tgt=None,
            model=MODEL,
            max_workers=MAX_WORKERS,
            prompt_function=prompt_function,
            partial_results=partial_results,
            sleep_time=setup.get("sleep_time", 0.0),
        )

        save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])
        plot_usage_histograms(tokens_usage, confidences, do_plot=False, do_print=False, suptitle=prompt_name)

        # Save token usage statistics
        token_usage_path = run_path / f"{SUBSET}_{MODEL}_{prompt_name}_token_usage{SUFFIX}.csv"
        token_stats = save_token_usage_stats(tokens_usage, token_usage_path)
        print(
            f"  {prompt_name}: {token_stats['total_input_tokens']:,} in / {token_stats['total_output_tokens']:,} out tokens"
        )

        try:
            predictions = get_predictions_with_gt(run_path, DATASET, SUBSET, MODEL, prompt_name, suffix=SUFFIX)
            save_analysis_results(
                predictions,
                print_results=False,
                plot_confusion_matrix=False,
                subtitle=f"{SUBSET}: {MODEL} {prompt_name} | ",
                cm_save_path=diagram_path,
                stats_path=stats_path,
            )
        except Exception as e:
            print(f"Error: {e}")

    store_run_metrics_df(PROMPT_FUNCTIONS, run_path, DATASET, SUBSET, MODEL, suffix=SUFFIX)